What pipeline_full.ipynb does and why it's structured this wayThe orchestrator notebook is the single entry point for headless execution. It doesn't import any src/ modules directly — its only job is to call Papermill to execute each stage notebook in sequence, pass parameters between them, handle failures gracefully, and produce a run report. This separation means you can always run any individual stage notebook interactively in JupyterLab without touching the orchestrator.The orchestrator also handles parameter sweeps — if you want to test multiple geometries or load cases in one make run, you define the sweep matrix in Cell 0 and the orchestrator loops over it, producing a separate output notebook and STL for each configuration.

Cell 0 — Parameters (tag: parameters)

In [ ]:
# Cell 0 — tagged: parameters
# This is the top-level Papermill entry point.
# Override any of these from the command line:
#   papermill pipeline_full.ipynb out.ipynb -p PART_NAME "bracket_v2"
#
# For a parameter sweep, set SWEEP_MODE=True and populate SWEEP_PARAMS.
import os
os.chdir("/workspace")

import sys
sys.path.insert(0, "/workspace")

PART_NAME_OVERRIDE = None   # set by pipeline_full.ipynb in sweep mode

# ── Single-run config ───────────────────────────────────────────────────────
PART_NAME          = "base_part"
PARAMS_JSON        = f"scad/{PART_NAME}_params.json"
OUTPUT_DIR         = "outputs"
KERNEL_NAME        = "fenics-pipeline"   # must match ipykernel install --name

# ── Per-stage overrides (passed to each stage notebook via Papermill) ───────
GEOMETRY_OVERRIDES = {}   # e.g. {"TIMEOUT_S": 180}
MESH_OVERRIDES     = {}   # e.g. {"ALGORITHM_3D": 10}
FEA_OVERRIDES      = {}   # e.g. {"YOUNGS_MODULUS_GPA": 70.0}
SIMP_OVERRIDES     = {}   # e.g. {"VOLUME_FRACTION": 0.3, "MAX_ITERATIONS": 150}
EXPORT_OVERRIDES   = {}   # e.g. {"VOXEL_RESOLUTION": 200}

# ── Sweep mode ──────────────────────────────────────────────────────────────
SWEEP_MODE   = False
SWEEP_PARAMS = [
    # Each dict is merged into the base overrides for one pipeline run
    # {"PART_NAME": "bracket_thin",  "SIMP_OVERRIDES": {"VOLUME_FRACTION": 0.3}},
    # {"PART_NAME": "bracket_thick", "SIMP_OVERRIDES": {"VOLUME_FRACTION": 0.5}},
]

# ── Failure handling ────────────────────────────────────────────────────────
ABORT_ON_STAGE_FAILURE = True   # False = continue sweep even if one run fails

Cell 1 — Imports and helpers

In [ ]:
# Cell 1 — Orchestration utilities
import json
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import papermill as pm


def run_stage(
    notebook_name: str,
    output_suffix: str,
    parameters: dict,
    kernel_name: str,
    executed_dir: Path,
) -> tuple[bool, float, Optional[str]]:
    """
    Execute a single stage notebook via Papermill.

    Returns (success, duration_s, error_message).
    Never raises — failures are captured and returned as error strings
    so the orchestrator can decide whether to abort or continue.

    Output notebooks are written to outputs/executed_nbs/ with a timestamp
    suffix so repeated runs don't overwrite each other.
    """
    nb_path  = Path("/workspace/notebooks") / notebook_name
    ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    out_path = executed_dir / f"{output_suffix}_{ts}.ipynb"

    print(f"\n{'─'*60}")
    print(f"  Stage: {notebook_name}")
    print(f"  Output: {out_path.name}")
    if parameters:
        print(f"  Params: {json.dumps(parameters, indent=4)}")
    print(f"{'─'*60}")

    t0 = time.perf_counter()
    try:
        pm.execute_notebook(
            str(nb_path),
            str(out_path),
            parameters=parameters,
            kernel_name=kernel_name,
            progress_bar=False,      # clean output in headless mode
            request_save_on_cell_execute=True,
        )
        duration = round(time.perf_counter() - t0, 2)
        print(f"  ✓ Completed in {duration}s")
        return True, duration, None

    except pm.PapermillExecutionError as e:
        duration = round(time.perf_counter() - t0, 2)
        msg = f"Cell {e.exec_count} raised {e.ename}: {e.evalue}"
        print(f"  ✗ FAILED after {duration}s")
        print(f"    {msg}")
        return False, duration, msg

    except Exception as e:
        duration = round(time.perf_counter() - t0, 2)
        msg = traceback.format_exc()
        print(f"  ✗ UNEXPECTED ERROR after {duration}s")
        print(f"    {msg}")
        return False, duration, msg

Cell 2 — Single run executor

In [ ]:
# Cell 2 — Execute a single pipeline run for one parameter configuration

def execute_pipeline(
    part_name: str,
    params_json: str,
    output_dir: str,
    kernel_name: str,
    geometry_overrides: dict,
    mesh_overrides: dict,
    fea_overrides: dict,
    simp_overrides: dict,
    export_overrides: dict,
    abort_on_failure: bool = True,
) -> dict:
    """
    Run all five stage notebooks in sequence for a single part configuration.
    Returns a run report dict.
    """
    executed_dir = Path(output_dir) / "executed_nbs"
    executed_dir.mkdir(parents=True, exist_ok=True)

    run_start = time.perf_counter()
    report = {
        "part_name":  part_name,
        "started_at": datetime.now(timezone.utc).isoformat(),
        "stages":     {},
        "success":    False,
        "error":      None,
    }

    # Stage definitions: (notebook filename, output suffix, extra parameters)
    stages = [
         (
            "01_geometry_openscad.ipynb",
            f"{part_name}_01_geometry",
            {
                "PART_NAME_OVERRIDE":   part_name,
                **geometry_overrides,
            },
        ),
        (
            "02_mesh_gmsh.ipynb",
            f"{part_name}_02_mesh",
            {
                "PART_NAME_OVERRIDE": part_name,
                **mesh_overrides,
            },
        ),
        (
            "03_fea_fenicsx.ipynb",
            f"{part_name}_03_fea",
            {
                "PART_NAME_OVERRIDE": part_name,
                **fea_overrides,
            },
        ),
        (
            "04_simp_optimization.ipynb",
            f"{part_name}_04_simp",
            {
                "PART_NAME_OVERRIDE": part_name,
                **simp_overrides,
            },
        ),
        (
            "05_stl_export.ipynb",
            f"{part_name}_05_export",
            {
                "PART_NAME_OVERRIDE": part_name,
                **export_overrides,
            },
        ),
    ]
    for notebook, suffix, params in stages:
        success, duration, error = run_stage(
            notebook_name=notebook,
            output_suffix=suffix,
            parameters=params,
            kernel_name=kernel_name,
            executed_dir=executed_dir,
        )
        report["stages"][notebook] = {
            "success":    success,
            "duration_s": duration,
            "error":      error,
        }
        if not success:
            report["error"] = f"{notebook} failed: {error}"
            if abort_on_failure:
                break

    # Check final STL exists as ultimate success signal
    stl_path = Path(output_dir) / "stl" / f"{part_name}_optimized.stl"
    all_stages_succeeded = all(
        s["success"] for s in report["stages"].values()
    )
    report["stl_path"] = str(stl_path) if stl_path.exists() else None
    report["success"]  = all_stages_succeeded and stl_path.exists()
    report["total_duration_s"] = round(time.perf_counter() - run_start, 2)
    report["completed_at"]  = datetime.now(timezone.utc).isoformat()

    return report

Cell 3 — Run the pipeline (single or sweep

In [4]:
# Cell 3 — Execute single run or parameter sweep

all_reports = []

if not SWEEP_MODE:
    # ── Single run ────────────────────────────────────────────────────────
    print(f"Starting pipeline: {PART_NAME}")
    print(f"Params:            {PARAMS_JSON}")
    print(f"Kernel:            {KERNEL_NAME}")

    report = execute_pipeline(
        part_name=PART_NAME,
        params_json=PARAMS_JSON,
        output_dir=OUTPUT_DIR,
        kernel_name=KERNEL_NAME,
        geometry_overrides=GEOMETRY_OVERRIDES,
        mesh_overrides=MESH_OVERRIDES,
        fea_overrides=FEA_OVERRIDES,
        simp_overrides=SIMP_OVERRIDES,
        export_overrides=EXPORT_OVERRIDES,
        abort_on_failure=ABORT_ON_STAGE_FAILURE,
    )
    all_reports.append(report)

else:
    # ── Sweep mode ────────────────────────────────────────────────────────
    print(f"Starting sweep: {len(SWEEP_PARAMS)} configurations")

    for i, sweep_config in enumerate(SWEEP_PARAMS):
        sweep_part = sweep_config.get("PART_NAME", f"{PART_NAME}_sweep_{i:02d}")
        print(f"\n{'═'*60}")
        print(f"  Sweep {i+1}/{len(SWEEP_PARAMS)}: {sweep_part}")
        print(f"{'═'*60}")

        report = execute_pipeline(
            part_name=sweep_part,
            params_json=PARAMS_JSON,
            output_dir=OUTPUT_DIR,
            kernel_name=KERNEL_NAME,
            geometry_overrides={**GEOMETRY_OVERRIDES,
                                 **sweep_config.get("GEOMETRY_OVERRIDES", {})},
            mesh_overrides={**MESH_OVERRIDES,
                            **sweep_config.get("MESH_OVERRIDES", {})},
            fea_overrides={**FEA_OVERRIDES,
                           **sweep_config.get("FEA_OVERRIDES", {})},
            simp_overrides={**SIMP_OVERRIDES,
                            **sweep_config.get("SIMP_OVERRIDES", {})},
            export_overrides={**EXPORT_OVERRIDES,
                              **sweep_config.get("EXPORT_OVERRIDES", {})},
            abort_on_failure=ABORT_ON_STAGE_FAILURE,
        )
        all_reports.append(report)

        if not report["success"] and ABORT_ON_STAGE_FAILURE:
            print(f"\n✗ Sweep aborted after failure in: {sweep_part}")
            break

Starting pipeline: base_part
Params:            scad/params.json
Kernel:            fenics-pipeline

────────────────────────────────────────────────────────────
  Stage: 01_geometry_openscad.ipynb
  Output: base_part_01_geometry_20260410_173446.ipynb
  Params: {
    "PARAMS_JSON": "scad/params.json",
    "PART_NAME_OVERRIDE": "base_part"
}
────────────────────────────────────────────────────────────
  ✗ UNEXPECTED ERROR after 0.0s
    Traceback (most recent call last):
  File "/dolfinx-env/lib/python3.12/site-packages/papermill/iorw.py", line 199, in read
    json.loads(path)
  File "/usr/lib/python3.12/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Exp

/tmp/ipykernel_1151/3943415882.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "started_at": datetime.utcnow().isoformat() + "Z",
/tmp/ipykernel_1151/1228652791.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts       = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
/tmp/ipykernel_1151/3943415882.py:87: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  report["completed_at"]  = datetime.utcnow().isoformat() + "Z"


Cell 4 — Run report

In [ ]:
# Cell 4 — Print run report and write JSON summary

print(f"\n{'═'*60}")
print("  PIPELINE RUN REPORT")
print(f"{'═'*60}")

total_success = sum(1 for r in all_reports if r["success"])
print(f"  Runs: {total_success}/{len(all_reports)} succeeded\n")

for report in all_reports:
    status = "✓" if report["success"] else "✗"
    print(f"  {status} {report['part_name']} "
          f"({report.get('total_duration_s', '?')}s)")

    for notebook, stage_data in report["stages"].items():
        s = "✓" if stage_data["success"] else "✗"
        print(f"      {s} {notebook:<40} {stage_data['duration_s']}s")
        if stage_data["error"]:
            print(f"        ↳ {stage_data['error'][:120]}")

    if report.get("stl_path"):
        print(f"      STL: {report['stl_path']}")
    print()

# Write JSON report
report_path = (Path(OUTPUT_DIR) / "executed_nbs" /
               f"pipeline_report_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.json")
report_path.write_text(json.dumps(all_reports, indent=2))
print(f"  Full report: {report_path}")

# Raise if any run failed — makes `make run` return a non-zero exit code
failed = [r for r in all_reports if not r["success"]]
if failed:
    names = [r["part_name"] for r in failed]
    raise RuntimeError(f"Pipeline failed for: {names}. "
                       f"Check executed notebooks in {OUTPUT_DIR}/executed_nbs/")

print("\n✅ All runs completed successfully")


════════════════════════════════════════════════════════════
  PIPELINE RUN REPORT
════════════════════════════════════════════════════════════
  Runs: 1/1 succeeded

  ✓ base_part (0.01s)
      ✗ 01_geometry_openscad.ipynb               0.0s
        ↳ Traceback (most recent call last):
  File "/dolfinx-env/lib/python3.12/site-packages/papermill/iorw.py", line 199, in re
      STL: outputs/stl/base_part_optimized.stl

  Full report: outputs/executed_nbs/pipeline_report_20260410_173450.json

✅ All runs completed successfully


/tmp/ipykernel_1151/2488541218.py:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f"pipeline_report_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.json")
